In [11]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

# -----------------------------
# Model paths
# -----------------------------
BINARY_MODEL_PATH = Path("../models/binary_stgcn_gait14_improved_stadedict.bin")
MULTI_MODEL_PATH  = Path("../models/multilabel_stgcn_state_dict.bin")

# -----------------------------
# Load state dicts
# -----------------------------
binary_state_dict = torch.load(str(BINARY_MODEL_PATH), map_location="cpu")
multi_state_dict  = torch.load(str(MULTI_MODEL_PATH), map_location="cpu")

# -----------------------------
# Define model architectures
# -----------------------------
class BinarySTGCN(nn.Module):
    def __init__(self, in_channels=3, num_joints=14, out_classes=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=1)
        self.pool = nn.AdaptiveAvgPool2d((1, num_joints))
        self.fc = nn.Linear(256 * num_joints, out_classes)
        
    def forward(self, x):
        N, C, T, V, M = x.shape
        x = x.view(N, C, T, V)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        x = x.flatten(1)
        return self.fc(x)

class MultiSTGCN(nn.Module):
    def __init__(self, in_channels=3, num_joints=14, out_classes=5):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=1)
        self.pool = nn.AdaptiveAvgPool2d((1, num_joints))
        self.fc = nn.Linear(256 * num_joints, out_classes)
        
    def forward(self, x):
        N, C, T, V, M = x.shape
        x = x.view(N, C, T, V)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        x = x.flatten(1)
        return self.fc(x)

# -----------------------------
# Initialize models and load state dicts
# -----------------------------
def clean_state_dict(state_dict):
    """Remove 'module.' prefix if present"""
    new_dict = {}
    for k, v in state_dict.items():
        if k.startswith('module.'):
            new_dict[k[7:]] = v
        else:
            new_dict[k] = v
    return new_dict

binary_model = BinarySTGCN(num_joints=14, out_classes=1)
multi_model  = MultiSTGCN(num_joints=14, out_classes=5)

binary_model.load_state_dict(clean_state_dict(binary_state_dict), strict=False)
multi_model.load_state_dict(clean_state_dict(multi_state_dict), strict=False)

binary_model.eval()
multi_model.eval()

# -----------------------------
# CSV path & GAIT joints
# -----------------------------
CSV_PATH = Path("../data/GAVD_data/MissionGate/output_normal/cljar9bqo00c43n6l2u5zmlru_left side_nan_Abnormal Gait_cerebral palsy_landmarks.csv")

GAIT_JOINTS = [
    2, 5,     # eyes
    11, 12,   # shoulders
    23, 24,   # hips
    25, 26,   # knees
    27, 28,   # ankles
    29, 30,   # heels
    31, 32    # foot index
]

ANOMALY_COLS = [
    "gait_anomaly_knee_sagittal_plane_abnormality",
    "gait_anomaly_trunk_balance_abnormality",
    "gait_anomaly_spatiotemporal_asymmetry",
    "gait_anomaly_hip_pelvic_control_deficit",
    "gait_anomaly_distal_foot_control_deficit",
]

# -----------------------------
# Load CSV
# -----------------------------
df = pd.read_csv(CSV_PATH)

num_frames = df['frame'].nunique()
num_channels = 3
num_joints = len(GAIT_JOINTS)
num_people = 1

X = np.zeros((1, num_channels, num_frames, num_joints, num_people), dtype=np.float32)

for frame_idx, frame in enumerate(sorted(df['frame'].unique())):
    frame_data = df[df['frame'] == frame]
    for _, row in frame_data.iterrows():
        landmark_id = int(row['landmark_id'])
        if landmark_id in GAIT_JOINTS:
            joint_idx = GAIT_JOINTS.index(landmark_id)
            X[0, 0, frame_idx, joint_idx, 0] = row['x_norm']
            X[0, 1, frame_idx, joint_idx, 0] = row['y_norm']
            X[0, 2, frame_idx, joint_idx, 0] = row['z_norm']

X_tensor = torch.from_numpy(X)

# -----------------------------
# Run models & convert outputs
# -----------------------------
with torch.no_grad():
    # Binary output
    binary_logits = binary_model(X_tensor)
    binary_prob   = torch.sigmoid(binary_logits)
    binary_label  = "abnormal" if binary_prob >= 0.5 else "normal"

    # Multi-label output only if abnormal
    if binary_label == "abnormal":
        multi_logits = multi_model(X_tensor)
        multi_probs  = torch.sigmoid(multi_logits)
        multi_labels = [col for i, col in enumerate(ANOMALY_COLS) if multi_probs[0, i] >= 0.5]
    else:
        multi_labels = []

# -----------------------------
# Print results
# -----------------------------
# -----------------------------
# Print results (clearer formatting)
# -----------------------------
print(f"Overall gait: {binary_label} (prob={binary_prob.item():.3f})")

if binary_label == "abnormal" and multi_labels:
    print("Detected anomalies:")
    for anomaly in multi_labels:
        print(f" - {anomaly}")
else:
    print("Detected anomalies: None")


Overall gait: abnormal (prob=0.502)
Detected anomalies:
 - gait_anomaly_knee_sagittal_plane_abnormality
 - gait_anomaly_trunk_balance_abnormality
 - gait_anomaly_spatiotemporal_asymmetry
